# Fair comparison of three search policies

This canonical Milestone 2 sweep compares BestOfN, Greedy, and Population on the same target `12`, moves `(-2, 1, 5)`, initial artifact `0`, primary score `-abs(value - 12)`, nine-evaluation/eight-trial budget, and root seeds `0` through `39`. Population has size `3` and tournament size `2`.

In [1]:
from pathlib import Path
import runpy

path = Path("benchmark.py")
if not path.exists():
    path = Path("examples/learn/policy_comparison/benchmark.py")
benchmark = runpy.run_path(path, run_name="__main__")
reports = benchmark["REPORTS"]

policy comparison: target=12 moves=(-2, 1, 5) seeds=0..39 budget=(evaluations=9, trials=8)
BestOfN: solved=0/40 scores={-7: 40} evaluations={9: 40} usage={(9, 8, 0, 0.0): 40} representative=(seed=0, lineage=[0, 5])
Greedy: solved=15/40 scores={-9: 1, -8: 1, -5: 3, -4: 2, -3: 3, -2: 3, -1: 12, 0: 15} evaluations={5: 1, 6: 7, 7: 4, 9: 28} usage={(5, 4, 0, 0.0): 1, (6, 5, 0, 0.0): 7, (7, 6, 0, 0.0): 4, (9, 8, 0, 0.0): 28} representative=(seed=1, lineage=[0, 5, 10, 11, 12])
Population: solved=14/40 scores={-10: 1, -9: 1, -8: 2, -6: 2, -5: 1, -4: 3, -3: 1, -2: 5, -1: 10, 0: 14} evaluations={5: 1, 6: 2, 7: 2, 8: 6, 9: 29} usage={(5, 4, 0, 0.0): 1, (6, 5, 0, 0.0): 2, (7, 6, 0, 0.0): 2, (8, 7, 0, 0.0): 6, (9, 8, 0, 0.0): 29} representative=(seed=0, lineage=[0, 5, 10, 11, 12])


## What the lineages mean

BestOfN always expands zero, so `[0, 5]` is one spoke from a star. Its result is also a proof: independent one-step successors are limited to `{-2, 1, 5}`, making `5` and score `-7` the best possible. Greedy's representative is the accepted chain `[0, 5, 10, 11, 12]`. Population creates a tree through tournament expansion and retention; `[0, 5, 10, 11, 12]` is the winning branch of that larger tree.

In [2]:
topologies = {
    "BestOfN": "star spoke",
    "Greedy": "chain",
    "Population": "winning tree branch",
}
for report in reports:
    print(
        f"{report.policy}: {topologies[report.policy]} "
        f"{list(report.representative_lineage)} "
        f"(seed {report.representative_seed})"
    )

BestOfN: star spoke [0, 5] (seed 0)
Greedy: chain [0, 5, 10, 11, 12] (seed 1)
Population: winning tree branch [0, 5, 10, 11, 12] (seed 0)


## Evidence boundary

This is a **small deterministic illustration in the locked project environment**, not broad scientific evidence and not a cross-version CPython guarantee. The user proposer continues to use `random.Random.choice` through `context.rng`, matching the existing accepted example contract. Local callable comparison uses `module:qualname` declaration identity; it does not prove behavioral equivalence.

In [3]:
import json

evidence = {}
for report in reports:
    evidence[report.policy] = {
        "solve_rate": [report.solve_count, report.run_count],
        "scores": list(report.score_distribution),
        "evaluations": list(report.evaluation_distribution),
        "usage": [[list(key), count] for key, count in report.usage_distribution],
        "representative_seed": report.representative_seed,
        "lineage": list(report.representative_lineage),
    }
print(json.dumps(evidence, sort_keys=True))

{"BestOfN": {"evaluations": [[9, 40]], "lineage": [0, 5], "representative_seed": 0, "scores": [[-7, 40]], "solve_rate": [0, 40], "usage": [[[9, 8, 0, 0.0], 40]]}, "Greedy": {"evaluations": [[5, 1], [6, 7], [7, 4], [9, 28]], "lineage": [0, 5, 10, 11, 12], "representative_seed": 1, "scores": [[-9, 1], [-8, 1], [-5, 3], [-4, 2], [-3, 3], [-2, 3], [-1, 12], [0, 15]], "solve_rate": [15, 40], "usage": [[[5, 4, 0, 0.0], 1], [[6, 5, 0, 0.0], 7], [[7, 6, 0, 0.0], 4], [[9, 8, 0, 0.0], 28]]}, "Population": {"evaluations": [[5, 1], [6, 2], [7, 2], [8, 6], [9, 29]], "lineage": [0, 5, 10, 11, 12], "representative_seed": 0, "scores": [[-10, 1], [-9, 1], [-8, 2], [-6, 2], [-5, 1], [-4, 3], [-3, 1], [-2, 5], [-1, 10], [0, 14]], "solve_rate": [14, 40], "usage": [[[5, 4, 0, 0.0], 1], [[6, 5, 0, 0.0], 2], [[7, 6, 0, 0.0], 2], [[8, 7, 0, 0.0], 6], [[9, 8, 0, 0.0], 29]]}}


In [4]:
greedy = next(report for report in reports if report.policy == "Greedy")
print(f"Greedy solved: {greedy.solve_count}/{greedy.run_count}")
print("Greedy scores:", dict(greedy.score_distribution))

Greedy solved: 15/40
Greedy scores: {-9: 1, -8: 1, -5: 3, -4: 2, -3: 3, -2: 3, -1: 12, 0: 15}
